# TP5. Comprendre la crédibilité actuarielle (Bühlmann)

## La grande question de ce TP

Un assureur a 20 catégories socioprofessionnelles (CSP) dans son portefeuille.
Certaines comptent 4000 assurés, d'autres seulement 7.
Comment intégrer **toutes** les CSP dans la tarification, sans sur-apprendre sur les petites ?

Trois réponses possibles que vous allez comparer :

| Approche | Idée | Inconvénient |
|---|---|---|
| **M0. Regrouper** | « n<100 → CSP_autre » | On perd l'information, seuil arbitraire |
| **M1. Crédibilité Bühlmann** | Z·moyenne_groupe + (1−Z)·moyenne_globale | Z calibré par les données, mais fonctionne-t-il ? |
| **M2. Tout garder (FE)** | 20 coefficients distincts | Bruit énorme sur les petites CSP |

## Ce que vous devez savoir faire à la fin du TP

1. Expliquer **pourquoi** $Z = \dfrac{n}{n+k}$, pas juste l'utiliser
2. Calibrer $k$ à partir des données (variance bruit / variance signal)
3. Reconnaître **quand** la crédibilité aide, et **quand** elle ne sert à rien
4. Choisir la bonne maille de crédibilité (CSP seule vs CSP × Garage)

## Organisation du TP

### Partie A. Comprendre la crédibilité sur données synthétiques
On crée nous-mêmes un faux portefeuille (8 régions, vraie fréquence connue), pour voir Bühlmann à l'œuvre dans un cas où **on sait quelle est la bonne réponse**.

### Partie B. Application sur freMPL (données réelles)
On rejoue exactement la même mécanique sur le vrai dataset, où la vérité n'est pas observable, et on compare 7 modèles.


## Aparté important. Deux usages très différents de la crédibilité

Quand un cours d'actuariat dit "crédibilité = expérience perso + collectif", il pense souvent au **Bonus-Malus**. Mais la même formule sert à autre chose, et c'est cet autre usage qu'on étudie dans ce TP. Pour éviter la confusion, il faut bien distinguer :

### Usage 1. Tarification a posteriori (Bonus-Malus, *experience rating*)

$$\hat\mu_{\text{assuré}} = Z \cdot \underbrace{\bar{x}_{\text{ses sinistres passés}}}_{\text{perso}} + (1-Z) \cdot \underbrace{\bar{x}_{\text{tarif a priori}}}_{\text{collectif}}$$

- L'unité d'observation est **un assuré observé sur plusieurs années**.
- $n$ = nombre d'années d'historique de l'assuré.
- "Expérience" est vraiment personnelle, ce sont ses propres sinistres.
- Cible du shrinkage : le tarif a priori (issu d'un GLM par exemple).

### Usage 2. Tarification segmentée, classes à faible volume (le sujet de ce TP)

$$\hat\mu_g = Z_g \cdot \underbrace{\bar{x}_g}_{\substack{\text{expérience}\\\text{de la classe}}} + (1-Z_g) \cdot \underbrace{\bar{x}_{\text{global}}}_{\text{portefeuille}}$$

- L'unité d'observation est **une classe tarifaire** (CSP, région, CSP × Garage…).
- $n_g$ = nombre d'assurés dans la classe.
- "Expérience" est celle du **groupe**, pas d'un individu.
- Cible du shrinkage : la moyenne du portefeuille.

### Tableau récapitulatif

| | Usage 1 : Bonus-Malus | Usage 2 : Segmentation |
|---|---|---|
| "Groupe" = | 1 assuré, plusieurs années | 1 classe, plusieurs assurés |
| $n$ | années d'historique | nb d'assurés dans la classe |
| Cible du shrinkage | tarif a priori | moyenne portefeuille |
| Équivalent ML | mise à jour bayésienne en ligne | **Ridge ($L_2$) sur effets fixes de classe** |
| Exemple typique | coefficient Bonus-Malus auto | M5 du TP : Bühlmann sur CSP × Garage |

### Pourquoi cette précision est essentielle pour le TP

L'usage 2, c'est mathématiquement **un GLM Ridge sur les effets fixes de classe**, avec une pénalité $\lambda = k$ calibrée à partir des résidus du portefeuille au lieu d'une cross-validation. C'est pour ça que dans la suite du TP, on compare frontalement crédibilité et GLM : ce ne sont pas deux mondes séparés, c'est la même opération de régularisation, juste avec deux vocabulaires (actuariel et ML).

Quand on lit "crédibilité = perso + collectif" dans un manuel, il faut donc traduire selon le contexte :
- en Bonus-Malus, "perso" est vraiment perso ;
- en segmentation (ce TP), "perso" veut dire **expérience de la classe** ($\bar{x}_g$), et "collectif" veut dire **moyenne portefeuille** ($\bar{x}_{\text{global}}$).


# Partie A. Crédibilité sur données synthétiques

## A.1. Le problème intuitif

Imaginez une compagnie d'assurance auto avec **8 régions** de tailles très différentes :

| Région | Nb assurés | Vraie fréquence (que nous fixons) |
|---|---|---|
| Île-de-France | 3000 | 15% |
| PACA | 1500 | 20% |
| Bretagne | 800 | 10% |
| Normandie | 400 | 12% |
| Bourgogne | 150 | 18% |
| Auvergne | 60 | 8% |
| Corse | 25 | 22% |
| Lozère | 10 | 11% |

Comme c'est nous qui simulons les données, on connaît la vraie fréquence. **C'est l'avantage pédagogique de cette partie A** : on pourra mesurer l'erreur des différentes méthodes contre la vérité, ce qu'on ne pourra jamais faire sur des données réelles.

> Analogie : si vous lancez une pièce 10 fois et obtenez 8 faces, vous ne concluez pas que la pièce est truquée. Avec 10 000 lancers et 80 % de faces, là oui. C'est exactement le problème de la Lozère (n=10) contre l'Île-de-France (n=3000).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Simulation du portefeuille synthétique
regions_synth = {
 "Ile-de-France": {"n": 3000, "vraie_freq": 0.15},
 "PACA": {"n": 1500, "vraie_freq": 0.20},
 "Bretagne": {"n": 800, "vraie_freq": 0.10},
 "Normandie": {"n": 400, "vraie_freq": 0.12},
 "Bourgogne": {"n": 150, "vraie_freq": 0.18},
 "Auvergne": {"n": 60, "vraie_freq": 0.08},
 "Corse": {"n": 25, "vraie_freq": 0.22},
 "Lozere": {"n": 10, "vraie_freq": 0.11},
}

rows = []
for region, info in regions_synth.items():
 sinistres = np.random.binomial(1, info["vraie_freq"], size=info["n"])
 for s in sinistres:
 rows.append({"region": region, "sinistre": s})
df_synth_a = pd.DataFrame(rows)

# Statistiques par région
stats_a = (df_synth_a.groupby("region")["sinistre"]
.agg(n="count", freq_observee="mean")
.reset_index())
stats_a["vraie_freq"] = stats_a["region"].map(
 {r: v["vraie_freq"] for r, v in regions_synth.items()})
stats_a["erreur_brute"] = (stats_a["freq_observee"] - stats_a["vraie_freq"]).abs()
stats_a = stats_a.sort_values("n", ascending=False).reset_index(drop=True)

print(f"Total assures simules : {len(df_synth_a):,}")
print()
print("Frequence observee vs vraie frequence par region :")
print(stats_a.to_string(index=False, float_format=lambda x: f"{x:.1%}" if abs(x) < 1 else f"{x:.0f}"))

## A.2. Visualiser le problème

Les petits groupes ont une fréquence observée qui s'éloigne fortement de la vraie fréquence : c'est du bruit d'échantillonnage, pas un signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Barres : observe vs vrai
ax = axes[0]
x = np.arange(len(stats_a))
ax.bar(x - 0.2, stats_a["freq_observee"] * 100, 0.4,
 label="Observee (nos donnees)", color="#4C72B0")
ax.bar(x + 0.2, stats_a["vraie_freq"] * 100, 0.4,
 label="Vraie (inconnue en realite)", color="#DD8452", alpha=0.75)
ax.set_xticks(x)
ax.set_xticklabels(stats_a["region"], rotation=35, ha="right")
ax.set_ylabel("Frequence sinistre (%)")
ax.set_title("Frequence observee vs vraie frequence")
ax.legend()
for i, row in stats_a.iterrows():
 ax.text(i, max(row["freq_observee"], row["vraie_freq"]) * 100 + 0.5,
 f"n={row['n']}", ha="center", fontsize=8, color="gray")

# Erreur en fonction de n
ax2 = axes[1]
couleurs = ["red" if n < 200 else "orange" if n < 1000 else "green"
 for n in stats_a["n"]]
ax2.scatter(stats_a["n"], stats_a["erreur_brute"] * 100,
 c=couleurs, s=110, zorder=3)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.set_xlabel("Nombre d'assures dans la region")
ax2.set_ylabel("Erreur absolue d'estimation (pp)")
ax2.set_title("Plus le groupe est petit, plus l'erreur est grande")
ax2.set_xscale("log")
for _, row in stats_a.iterrows():
 ax2.annotate(row["region"], (row["n"], row["erreur_brute"] * 100),
 textcoords="offset points", xytext=(6, 0), fontsize=8)
patches_leg = [
 mpatches.Patch(color='green', label='n >= 1000'),
 mpatches.Patch(color='orange', label='200 <= n < 1000'),
 mpatches.Patch(color='red', label='n < 200'),
]
ax2.legend(handles=patches_leg)
plt.tight_layout()
plt.show()

print("Observation : Lozere (n=10) sous-estime de plus de 1 pp,")
print("Corse (n=25) surestime de plusieurs pp. Pas un signal, juste du bruit.")

## A.3. La formule de Bühlmann et son intuition

Pour chaque région $g$, on calcule une prime crédibilisée :

$$\hat{\mu}_g = Z_g \cdot \hat{f}_g + (1 - Z_g) \cdot \bar{f}, \quad Z_g = \frac{n_g}{n_g + k}$$

où :
- $\hat{f}_g$ est la fréquence observée dans la région
- $\bar{f}$ est la moyenne globale du portefeuille (stable)
- $Z_g$ est le **facteur de crédibilité**, compris entre 0 et 1
- $k$ est un paramètre à calibrer, défini par :

$$k = \frac{\text{variance du processus (bruit)}}{\text{variance des vraies fréquences (signal)}}$$

Intuition de $k$ :
- $k$ grand : signal entre groupes faible, on ne fait pas confiance à l'expérience locale, $Z$ reste petit
- $k$ petit : groupes vraiment différents, on fait vite confiance à l'expérience locale, $Z \to 1$ rapidement

In [ ]:
# Calibration de Buhlmann sur le portefeuille synthetique
mu_global_a = df_synth_a["sinistre"].mean()

# Variance du processus (bruit intra-region, approximation binomiale)
variance_processus_a = stats_a.apply(
 lambda row: row["freq_observee"] * (1 - row["freq_observee"]),
 axis=1
).mean()

# Variance structurelle (signal entre regions)
variance_structurelle_a = stats_a["freq_observee"].var()

k_a = variance_processus_a / variance_structurelle_a

print(f"Frequence globale du portefeuille : {mu_global_a:.1%}")
print(f"Variance processus (bruit) : {variance_processus_a:.5f}")
print(f"Variance structurelle (signal) : {variance_structurelle_a:.5f}")
print(f"k = bruit / signal : {k_a:.1f}")
print(f"\nInterpretation : il faut n = {k_a:.0f} assures pour Z = 50 %")
print(f" il faut n = {9*k_a:.0f} assures pour Z = 90 %")

# Application
stats_a["Z"] = stats_a["n"] / (stats_a["n"] + k_a)
stats_a["prime_credibilisee"] = (stats_a["Z"] * stats_a["freq_observee"]
 + (1 - stats_a["Z"]) * mu_global_a)
stats_a["erreur_credibilite"] = (stats_a["prime_credibilisee"]
 - stats_a["vraie_freq"]).abs()

print("\nResultat par region :")
cols = ["region", "n", "Z", "freq_observee", "prime_credibilisee", "vraie_freq",
 "erreur_brute", "erreur_credibilite"]
print(stats_a[cols].to_string(index=False,
 float_format=lambda x: f"{x:.1%}" if abs(x) < 1 else f"{x:.0f}"))

gain = (stats_a["erreur_brute"].sum() - stats_a["erreur_credibilite"].sum()) * 100
print(f"\nVariation totale d'erreur (brute - credibilite) : {gain:+.2f} pp cumules")
print("Lecture : sur ce tirage precis, Buhlmann aide certains groupes (Bourgogne,")
print("Auvergne) mais empire d'autres (Corse et Lozere avaient eu de la chance).")
print("En esperance sur tous les tirages possibles, Buhlmann minimise le MSE,")
print("mais sur UN tirage donne on peut perdre ponctuellement.")

## A.4. Bühlmann corrige les petits groupes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

regions_order = stats_a["region"].tolist()
x = np.arange(len(regions_order))

# Les 3 estimations cote a cote
ax = axes[0]
ax.bar(x - 0.25, stats_a["freq_observee"] * 100, 0.25,
 label="Brute (donnees)", color="#4C72B0")
ax.bar(x + 0.00, stats_a["prime_credibilisee"] * 100, 0.25,
 label="Credibilite Buhlmann", color="#55A868")
ax.bar(x + 0.25, stats_a["vraie_freq"] * 100, 0.25,
 label="Vraie (reference)", color="#DD8452", alpha=0.75)
ax.axhline(mu_global_a * 100, color="gray", linestyle="--", linewidth=1,
 label=f"Moyenne globale ({mu_global_a:.1%})")
ax.set_xticks(x)
ax.set_xticklabels(regions_order, rotation=35, ha="right")
ax.set_ylabel("Frequence sinistre (%)")
ax.set_title("Brute vs Credibilite vs Vraie frequence")
ax.legend(fontsize=9)
for i, row in stats_a.iterrows():
 ax.text(i - 0.25, row["freq_observee"] * 100 + 0.3, f"Z={row['Z']:.0%}",
 ha="center", fontsize=7, color="#4C72B0")

# Reduction de l'erreur
ax2 = axes[1]
ax2.bar(x - 0.2, stats_a["erreur_brute"] * 100, 0.4,
 label="Erreur brute", color="#C44E52")
ax2.bar(x + 0.2, stats_a["erreur_credibilite"] * 100, 0.4,
 label="Erreur credibilite", color="#55A868")
ax2.set_xticks(x)
ax2.set_xticklabels(regions_order, rotation=35, ha="right")
ax2.set_ylabel("Erreur absolue (pp)")
ax2.set_title("Effet de Buhlmann sur l'erreur, par taille de groupe")
ax2.legend()
plt.tight_layout()
plt.show()

print("Lecture :")
print(" - Pour Lozere (n=10, Z faible), Buhlmann ramene vers la moyenne globale.")
print(" - Pour Ile-de-France (n=3000, Z proche de 1), Buhlmann ne change quasi rien.")
print(" - Sur les groupes ou la mesure brute etait tres loin de la verite")
print(" (Bourgogne, Auvergne), Buhlmann reduit fortement l'erreur.")
print(" - Sur ceux ou la mesure brute etait proche par chance (Corse, Lozere),")
print(" Buhlmann peut empirer ponctuellement : c'est le prix de la stabilite.")

## A.5. À retenir de la partie A

Sur ce portefeuille synthétique, Bühlmann fonctionne **comme prévu** :

| | Méthode brute | Crédibilité Bühlmann |
|---|---|---|
| Petits groupes (Lozère, Corse) | Estimation chaotique | Ramenée vers la moyenne, stable |
| Grands groupes (IdF, PACA) | Correcte | Z proche de 1, quasiment identique |
| Paramètre clé | aucun | $k$ = bruit / signal |

Trois messages :
1. $Z$ proche de 0 : peu de données, on utilise surtout la moyenne globale.
2. $Z$ proche de 1 : beaucoup de données, on fait confiance à l'expérience propre.
3. $k$ mesure combien de données il faut pour gagner la confiance, et dépend du portefeuille.

C'est exactement le mécanisme (n)$ qu'on retrouve dans le **bonus-malus** auto (cf. aparté en début de TP, Usage 1) : la compagnie vous croit de plus en plus au fil du temps. Ici, on l'applique à des **classes d'assurés** plutôt qu'à un assuré individuel (Usage 2).

> **Teaser pour la partie B** : si les 20 CSP de freMPL sont **toutes** au-dessus de n=500, qu'est-ce qu'on peut attendre de Bühlmann appliqué à CSP seule ? On y répond formellement en Question 1.

## Étape B.1. Imports et chargement des données brutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Binomial
import warnings
warnings.filterwarnings("ignore")

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Chargement dataset brut (avec target)
df_raw = pd.read_csv("df_raw_with_target.csv")
print(f"Dataset : {len(df_raw):,} lignes | Taux sinistre : {df_raw['target'].mean():.1%}")
print(f"Colonnes : {df_raw.columns.tolist()}")

## Étape B.2. Preprocessing (identique au TP2)

On rejoue exactement le même pipeline que le TP2 :
- Variables numériques nettoyées
- Split 70/30 stratifié (seed=42)
- **Différence** : on garde `SocioCateg` **brute** (les 20 modalités) pour pouvoir appliquer la crédibilité

In [ ]:
df = df_raw.copy()

# --- Nettoyage variables texte → numérique (TP2) ---
def midpoint(s):
 """Extrait le midpoint d'un intervalle du type '1-3' ou '5'."""
 s = str(s).strip()
 if '-' in s:
 parts = s.split('-')
 try:
 return (float(parts[0]) + float(parts[1])) / 2
 except:
 return np.nan
 try:
 return float(s)
 except:
 return np.nan

df["VehAge_num"] = df["VehAge"].apply(midpoint)
df["VehMaxSpeed_num"] = df["VehMaxSpeed"].apply(
 lambda s: float(str(s).split('-')[0]) if pd.notna(s) and '-' in str(s) else np.nan)

# Garage : NA → "Unknown"
df["Garage"] = df["Garage"].fillna("Unknown")

# MariStat binaire
df["MariAlone"] = (df["MariStat"] == "Alone").astype(int)

# Gender binaire
df["Gender_F"] = (df["Gender"] == "Female").astype(int)

# --- Variables retenues pour le GLM (sans SocioCateg pour l'instant) ---
VARS_NUM = ["LicAge", "DrivAge", "BonusMalus", "RiskVar", "HasKmLimit",
 "VehAge_num", "VehMaxSpeed_num"]
VARS_CAT = ["VehUsage", "VehBody", "VehPrice", "VehEngine",
 "VehEnergy", "VehClass", "Garage"]
VARS_BIN = ["Gender_F", "MariAlone"]
TARGET = "target"
CSP_COL = "SocioCateg" # gardée brute pour la crédibilité

# Suppression lignes avec NA sur variables modèle
cols_needed = VARS_NUM + VARS_CAT + VARS_BIN + [TARGET, CSP_COL]
df_clean = df[cols_needed].dropna().reset_index(drop=True)
print(f"Après nettoyage : {len(df_clean):,} lignes ({len(df_raw)-len(df_clean)} supprimées)")
print(f"Taux sinistre : {df_clean[TARGET].mean():.1%}")

# --- Split 70/30 stratifié seed=42 (identique TP2) ---
train_idx, test_idx = train_test_split(
 df_clean.index, test_size=0.30, random_state=42,
 stratify=df_clean[TARGET])

df_train = df_clean.loc[train_idx].reset_index(drop=True)
df_test = df_clean.loc[test_idx].reset_index(drop=True)

print(f"\nTrain : {len(df_train):,} | Test : {len(df_test):,}")
print(f"Taux sinistre train : {df_train[TARGET].mean():.1%} | test : {df_test[TARGET].mean():.1%}")

## Étape B.3. Fonctions d'évaluation (identiques au TP4)

On définit les métriques actuarielles standard.

In [ ]:
def gini(y_true, y_score):
 return 2 * roc_auc_score(y_true, y_score) - 1

def lift_decile1(y_true, y_score):
 """Lift au 1er décile : combien de fois plus de sinistres dans le top 10% prédit."""
 df_tmp = pd.DataFrame({"y": y_true, "score": y_score})
 df_tmp = df_tmp.sort_values("score", ascending=False).reset_index(drop=True)
 n = len(df_tmp)
 top10 = df_tmp.iloc[:int(n * 0.10)]
 taux_top10 = top10["y"].mean()
 taux_global = df_tmp["y"].mean()
 return taux_top10 / taux_global

def evaluer(nom, y_true, y_score):
 """Retourne un dict de métriques pour un modèle."""
 return {
 "Modèle": nom,
 "AUC": roc_auc_score(y_true, y_score),
 "Gini": gini(y_true, y_score),
 "Brier": brier_score_loss(y_true, y_score),
 "Lift D1": lift_decile1(y_true, y_score),
 }

print("Fonctions d'évaluation définies : AUC, Gini, Brier Score, Lift Décile 1")

## Étape B.3bis. L'intuition de la crédibilité (AVANT les formules)

### Le problème en 1 phrase
On observe une fréquence de sinistre $\hat{f}_g$ dans la CSP $g$ (taille $n_g$).
**Faut-il faire confiance** à cette fréquence ?

### Réponse intuitive
- Si $n_g = 4000$ → oui, $\hat{f}_g$ est précise
- Si $n_g = 7$ → non, c'est du bruit
- Entre les deux → on **pondère** entre $\hat{f}_g$ (le groupe) et $\bar{f}$ (la moyenne globale, plus stable)

C'est exactement ce que dit Bühlmann :
$$\hat{p}_g = Z_g \cdot \hat{f}_g + (1 - Z_g) \cdot \bar{f}, \quad Z_g = \frac{n_g}{n_g + k}$$

Le paramètre **$k$** contrôle « à partir de combien d'observations je commence à faire confiance ».

### Question 1. Avant de coder, premier diagnostic
> Si la **variance entre CSP** (signal) est faible et la **variance dans une CSP** (bruit) est forte,
> $k$ doit-il être **grand** ou **petit** ? Pourquoi ?


In [ ]:
# Visualisons Z(n) pour différentes valeurs de k
# Plus k est grand, plus il faut d'observations pour faire confiance au groupe

n_grid = np.arange(1, 5001)
fig, ax = plt.subplots(figsize=(11, 5))

annot_y = {5: 0.20, 50: 0.35, 500: 0.55} # positions verticales pour éviter le chevauchement

for k, couleur in [(5, "#2196F3"), (50, "#FF9800"), (500, "#F44336")]:
 Z = n_grid / (n_grid + k)
 ax.plot(n_grid, Z, label=f"k = {k}", color=couleur, linewidth=2)
 ax.scatter([k], [0.5], color=couleur, s=80, zorder=5)
 ax.annotate(f"Z=50% à n={k}",
 xy=(k, 0.5),
 xytext=(k + 300, annot_y[k]),
 fontsize=9, color=couleur,
 arrowprops=dict(arrowstyle="->", color=couleur, lw=0.8, alpha=0.6))

ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axhline(1.0, color="gray", linestyle=":", linewidth=0.8, alpha=0.4)
ax.set_xlabel("Taille du groupe n_g")
ax.set_ylabel("Facteur de crédibilité Z = n / (n + k)")
ax.set_title("Comment Z grandit avec n, pour 3 valeurs de k", fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10, loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Lecture :")
print(" k = 5 → on fait vite confiance au groupe (Z=50% dès n=5)")
print(" k = 50 → il faut au moins n=50 pour 50% de confiance")
print(" k = 500 → on n'a presque jamais confiance (Z<50% même à n=300)")
print("\n→ Plus le BRUIT dans les groupes est fort par rapport au SIGNAL")
print(" (vraie différence entre CSP), plus k est grand.")
print("\n→ Notre boulot dans M1 sera d'ESTIMER k à partir des résidus du GLM.")


## Étape B.4. M0 : GLM baseline (reproduit TP3)

CSP regroupées : toutes les CSP avec n < 100 sur le **train** → "CSP_autre". 
C'est la décision prise dans le TP2, c'est notre référence.

In [ ]:
# --- Regroupement CSP (calculé sur train uniquement pour éviter le data leakage) ---
csp_counts_train = df_train[CSP_COL].value_counts()
csp_grandes = csp_counts_train[csp_counts_train >= 100].index.tolist()

def grouper_csp(s):
 return s if s in csp_grandes else "CSP_autre"

df_train["CSP_g"] = df_train[CSP_COL].map(grouper_csp)
df_test["CSP_g"] = df_test[CSP_COL].map(grouper_csp)

print("CSP conservées (n≥100 sur train) :")
print(sorted(csp_grandes))
print(f"\nCSP regroupées en 'CSP_autre' : {[c for c in df_train[CSP_COL].unique() if c not in csp_grandes]}")
print(f"\nRépartition CSP_g train :\n{df_train['CSP_g'].value_counts()}")

# --- Formule GLM M0 ---
formula_m0 = (
 "target ~ C(CSP_g) + C(VehUsage) + C(VehBody) + C(VehPrice) "
 "+ C(VehEngine) + C(VehEnergy) + C(VehClass) + C(Garage) "
 "+ Gender_F + MariAlone + LicAge + DrivAge + BonusMalus "
 "+ RiskVar + HasKmLimit + VehAge_num + VehMaxSpeed_num"
)

# Ajustement sur train
m0 = glm(formula_m0, data=df_train, family=Binomial()).fit()
print(f"\nM0 ajusté, AIC : {m0.aic:.1f} | Nb paramètres : {m0.params.shape[0]}")

# Prédictions
df_train["pred_m0"] = m0.predict(df_train)
df_test["pred_m0"] = m0.predict(df_test)

res_m0 = evaluer("M0, GLM baseline (CSP_autre)", df_test[TARGET], df_test["pred_m0"])
print(f"\n{'='*50}")
print(f"M0, Test : AUC={res_m0['AUC']:.4f} | Gini={res_m0['Gini']:.4f} | "
 f"Brier={res_m0['Brier']:.4f} | Lift D1={res_m0['Lift D1']:.2f}x")

### Question 2. Critique du regroupement M0

Regardez la liste des CSP qui ont été fusionnées en `CSP_autre`.

**a)** Pour ces CSP, à quoi M0 va-t-il prédire la fréquence de sinistre ?
*(Aide : c'est la moyenne de **toutes** les petites CSP mélangées)*

**b)** Si CSP65 a en réalité une fréquence de **15%** et CSP46 une fréquence de **3%**, mais qu'on les mélange dans `CSP_autre` avec une moyenne de **7%** :
- Est-ce un problème pour le **classement** des risques (AUC) ?
- Est-ce un problème pour la **tarification** individuelle ?

**c)** Pourquoi avoir choisi le seuil **n<100** plutôt que 50 ou 200 ?

> Ces questions reviendront dans la **synthèse finale**. Notez vos hypothèses.


## Étape B.5. M1 : GLM sans CSP + Crédibilité Bühlmann

### Pipeline
1. GLM ajusté **sans** `SocioCateg` → capture les effets individuels (âge, BM, véhicule)
2. Calcul des **résidus par CSP** sur le train : `residu_g = freq_observee_g - pred_glm_g`
3. Calibration de **k** sur le train (bruit / signal entre CSP)
4. Correction crédibilisée : `Z_g = n_g / (n_g + k)` → **appliquée sur le test**

**Avantage clé** : les 20 CSP sont toutes utilisées, même CSP65 (7 individus), mais avec Z≈0 elle ne bouge presque pas de la prédiction GLM.

### Exercice 1. Bühlmann à la main sur une seule CSP

Avant d'écrire le code complet, faisons-le **pas à pas** pour une seule CSP : **CSP50** (la plus grande, ~4000 individus).

**Idée intuitive** :
1. On a un GLM **sans CSP** qui prédit une moyenne $\bar{p}_g$ pour les gens de CSP50.
2. On observe une fréquence **réelle** $\hat{f}_g$ sur ces 4000 personnes.
3. L'écart $\hat{f}_g - \bar{p}_g$ est-il un signal vrai (CSP50 est plus/moins risquée que ce que dit le GLM) ou du bruit (échantillonnage) ?
4. Bühlmann répond : on garde **une fraction Z** de cet écart.

La cellule suivante fait ce calcul **uniquement pour CSP50**, pour bien voir tous les nombres.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Étape A, Ajuster un GLM SANS la variable CSP (= modèle "moyenne globale" par profil)
# ─────────────────────────────────────────────────────────────────────────────
formula_pedago = (
 "target ~ C(VehUsage) + C(VehBody) + C(VehPrice) "
 "+ C(VehEngine) + C(VehEnergy) + C(VehClass) + C(Garage) "
 "+ Gender_F + MariAlone + LicAge + DrivAge + BonusMalus "
 "+ RiskVar + HasKmLimit + VehAge_num + VehMaxSpeed_num"
)
glm_pedago = glm(formula_pedago, data=df_train, family=Binomial()).fit()
df_train["pred_sans_csp"] = glm_pedago.predict(df_train)

# ─────────────────────────────────────────────────────────────────────────────
# Étape B, Zoom sur CSP50 (la plus grande)
# ─────────────────────────────────────────────────────────────────────────────
csp_focus = "CSP50"
g50 = df_train[df_train[CSP_COL] == csp_focus]

n_g = len(g50)
freq_obs = g50[TARGET].mean() # fréquence réelle dans CSP50
pred_glm = g50["pred_sans_csp"].mean() # ce que prédit le GLM sans CSP
residu = freq_obs - pred_glm # écart à expliquer

print(f"=== Analyse de la CSP « {csp_focus} » ===")
print(f" Taille n_g = {n_g}")
print(f" Fréquence observée = {freq_obs:.2%}")
print(f" Prédiction GLM sans CSP = {pred_glm:.2%}")
print(f" → Résidu (écart) = {residu:+.2%}")
print()

# ─────────────────────────────────────────────────────────────────────────────
# Étape C, Bruit vs Signal (intuition Bühlmann)
# ─────────────────────────────────────────────────────────────────────────────
# Pour CALIBRER k, il nous faut comparer :
# - le bruit attendu DANS chaque CSP (variance binomiale ≈ p(1-p)/n)
# - le signal entre CSP (var. des résidus entre groupes)
# On fait ça sur TOUTES les CSP :

residus_all = (df_train.groupby(CSP_COL)
.apply(lambda x: pd.Series({
 "n": len(x),
 "freq_obs": x[TARGET].mean(),
 "pred_glm": x["pred_sans_csp"].mean(),
 "residu": x[TARGET].mean() - x["pred_sans_csp"].mean(),
 })).reset_index())

# Bruit : variance binomiale moyenne (chaque CSP contribue p(1-p))
bruit = residus_all["residu"].apply(lambda r: abs(r) * (1 - abs(r))).mean()
# Signal : variance des écarts entre CSP
signal = residus_all["residu"].var()
k_demo = bruit / signal

print(f"=== Calibration de k (sur TOUTES les CSP) ===")
print(f" Variance bruit (moyenne intra-CSP) = {bruit:.5f}")
print(f" Variance signal (entre CSP) = {signal:.5f}")
print(f" k = bruit / signal = {k_demo:.2f}")
print()

# ─────────────────────────────────────────────────────────────────────────────
# Étape D, Z et correction pour CSP50
# ─────────────────────────────────────────────────────────────────────────────
Z_50 = n_g / (n_g + k_demo)
correction = Z_50 * residu
pred_buhlmann = pred_glm + correction

print(f"=== Crédibilité Bühlmann pour {csp_focus} ===")
print(f" Z = n / (n + k) = {n_g} / ({n_g} + {k_demo:.2f}) = {Z_50:.4f}")
print(f" Correction = Z × résidu = {Z_50:.4f} × {residu:+.2%} = {correction:+.4%}")
print(f" Prédiction finale = GLM + correction = {pred_glm:.2%} + {correction:+.4%} = {pred_buhlmann:.2%}")
print()
print(f" → Comparaison à la fréquence observée ({freq_obs:.2%}) :")
print(f" GLM sans CSP = {pred_glm:.2%} (écart {(pred_glm-freq_obs)*100:+.2f} pp)")
print(f" GLM + Bühlmann = {pred_buhlmann:.2%} (écart {(pred_buhlmann-freq_obs)*100:+.2f} pp)")


In [ ]:
# --- 5a. GLM sans CSP ---
formula_m1_glm = (
 "target ~ C(VehUsage) + C(VehBody) + C(VehPrice) "
 "+ C(VehEngine) + C(VehEnergy) + C(VehClass) + C(Garage) "
 "+ Gender_F + MariAlone + LicAge + DrivAge + BonusMalus "
 "+ RiskVar + HasKmLimit + VehAge_num + VehMaxSpeed_num"
)

m1_glm = glm(formula_m1_glm, data=df_train, family=Binomial()).fit()
print(f"GLM sans CSP ajusté, AIC : {m1_glm.aic:.1f} | Nb paramètres : {m1_glm.params.shape[0]}")

df_train["pred_m1_glm"] = m1_glm.predict(df_train)
df_test["pred_m1_glm"] = m1_glm.predict(df_test)

# --- 5b. Résidus par CSP sur le TRAIN ---
residus_train = (df_train.groupby(CSP_COL)
.apply(lambda g: pd.Series({
 "n_train": len(g),
 "freq_obs_train": g[TARGET].mean(),
 "pred_glm_train": g["pred_m1_glm"].mean(),
 "residu_train": g[TARGET].mean() - g["pred_m1_glm"].mean(),
 }))
.reset_index())

print(f"\n=== Résidus par CSP (calculés sur TRAIN uniquement) ===")
print(residus_train.sort_values("n_train", ascending=False)
.to_string(index=False, float_format=lambda x: f"{x:.1%}" if abs(x) < 1 else f"{x:.0f}"))

# --- 5c. Calibration k sur le TRAIN ---
variance_bruit = residus_train["residu_train"].apply(lambda r: abs(r) * (1 - abs(r))).mean()
variance_signal = residus_train["residu_train"].var()
k_train = variance_bruit / variance_signal if variance_signal > 1e-9 else 50

residus_train["Z"] = residus_train["n_train"] / (residus_train["n_train"] + k_train)
residus_train["correction_train"] = residus_train["Z"] * residus_train["residu_train"]

print(f"\nParamètre k calibré sur train : {k_train:.2f}")
print(f"→ Il faut n ≈ {k_train:.0f} assurés pour Z = 50%")
print(residus_train[[CSP_COL, "n_train", "Z", "residu_train", "correction_train"]]
.sort_values("n_train", ascending=False)
.to_string(index=False, float_format=lambda x: f"{x:.1%}" if abs(x) < 1 else f"{x:.0f}"))

In [ ]:
# --- 5d. Application sur le TEST ---
# La correction est celle apprise sur le train → pas de data leakage
csp_to_correction = residus_train.set_index(CSP_COL)["correction_train"].to_dict()
csp_to_Z = residus_train.set_index(CSP_COL)["Z"].to_dict()

# CSP inconnue dans le test (si une CSP n'apparaît pas en train) → correction = 0
df_test["correction_cred"] = df_test[CSP_COL].map(csp_to_correction).fillna(0)
df_test["pred_m1"] = (df_test["pred_m1_glm"] + df_test["correction_cred"]).clip(0.001, 0.999)

# Idem sur le train (pour vérification)
df_train["correction_cred"] = df_train[CSP_COL].map(csp_to_correction).fillna(0)
df_train["pred_m1"] = (df_train["pred_m1_glm"] + df_train["correction_cred"]).clip(0.001, 0.999)

res_m1 = evaluer("M1, GLM + Crédibilité Bühlmann", df_test[TARGET], df_test["pred_m1"])
print(f"M1, Test : AUC={res_m1['AUC']:.4f} | Gini={res_m1['Gini']:.4f} | "
 f"Brier={res_m1['Brier']:.4f} | Lift D1={res_m1['Lift D1']:.2f}x")

### Question 3. Le résultat de M1 est-il surprenant ?

**M1 n'améliore PAS M0** (vous le verrez : AUC ≈ pareil voire un peu moins).

**a)** Regardez la colonne `Z` dans le tableau ci-dessus. Combien de CSP ont $Z > 95\%$ ?

**b)** Si Z≈1 pour presque toutes les CSP, alors la correction $Z \cdot \text{résidu}$ vaut **résidu** entier.
Mais le résidu est calculé sur le **train**. Que se passe-t-il sur le **test** si la fréquence d'une CSP diffère un peu (bruit d'échantillonnage) ?

**c)** **Diagnostic** : Bühlmann est censé aider quand **k ≫ n**. Ici on a $k \approx 2.65$ et n entre 7 et 4000. Pourquoi cette calibration nous donne-t-elle un Z élevé partout ?

> Indice : la **vraie** différence entre CSP (variance signal) est faible, donc Bühlmann conclut « les CSP sont presque toutes pareilles, je leur fais confiance ».
> Ce constat est la **clé** des améliorations M4 et M5 plus loin.


## Étape B.6. M2 : GLM avec toutes les CSP en effets fixes

C'est le GLM "naïf" qui inclut toutes les 20 CSP avec Z=1 partout. 
Pour les petites CSP (n=7), le coefficient sera très bruité → on s'attend à de l'**overfitting**.

In [ ]:
formula_m2 = (
 "target ~ C(SocioCateg) + C(VehUsage) + C(VehBody) + C(VehPrice) "
 "+ C(VehEngine) + C(VehEnergy) + C(VehClass) + C(Garage) "
 "+ Gender_F + MariAlone + LicAge + DrivAge + BonusMalus "
 "+ RiskVar + HasKmLimit + VehAge_num + VehMaxSpeed_num"
)

m2 = glm(formula_m2, data=df_train, family=Binomial()).fit()
print(f"M2 ajusté, AIC : {m2.aic:.1f} | Nb paramètres : {m2.params.shape[0]}")

df_test["pred_m2"] = m2.predict(df_test).clip(0.001, 0.999)
df_train["pred_m2"] = m2.predict(df_train).clip(0.001, 0.999)

res_m2 = evaluer("M2, GLM toutes CSP (Z=1)", df_test[TARGET], df_test["pred_m2"])
print(f"M2, Test : AUC={res_m2['AUC']:.4f} | Gini={res_m2['Gini']:.4f} | "
 f"Brier={res_m2['Brier']:.4f} | Lift D1={res_m2['Lift D1']:.2f}x")

### Question 4. Avant de regarder l'étape 7, classez M0/M1/M2

Faites un **pronostic** AVANT d'exécuter la cellule de comparaison.

**a)** Qui est le meilleur en **AUC** (capacité à classer les risques) ? Pourquoi ?

**b)** Qui est le meilleur en **Brier Score** (calibration des probabilités) ? Pourquoi ?

**c)** Le `CSP65` (7 personnes) a-t-il un coefficient fiable dans M2 ? Que se passerait-il si on l'enlevait du train et qu'on le rencontrait en production ?

**d)** Si vous étiez actuaire chez un assureur, **lequel** des trois recommanderiez-vous ? 
*(piège : la réponse n'est pas forcément le meilleur AUC)*

> Notez votre pronostic dans une cellule (ou sur papier). On y revient en synthèse.


## Étape B.7. Comparaison finale : tableau et graphiques

In [ ]:
# --- Tableau comparatif ---
resultats = pd.DataFrame([res_m0, res_m1, res_m2])
resultats = resultats.set_index("Modèle")

print("=" * 75)
print("COMPARAISON DES 3 MODÈLES, SET DE TEST")
print("=" * 75)
print(resultats.to_string(float_format=lambda x: f"{x:.4f}"))
print()

# Meilleur modèle par métrique
for col in resultats.columns:
 if col == "Brier":
 best = resultats[col].idxmin()
 direction = "↓ (plus petit = meilleur)"
 else:
 best = resultats[col].idxmax()
 direction = "↑ (plus grand = meilleur)"
 print(f" Meilleur {col} {direction} → {best}")

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Comparaison M0 / M1 / M2, Set de test freMPL", fontsize=13, fontweight="bold")

modeles = [
 ("M0, Baseline (CSP_autre)", df_test["pred_m0"], "#4C72B0"),
 ("M1, GLM + Crédibilité", df_test["pred_m1"], "#55A868"),
 ("M2, GLM toutes CSP (Z=1)", df_test["pred_m2"], "#CCB974"),
]

# --- Graphique 1 : Courbes ROC ---
ax = axes[0]
for nom, preds, col in modeles:
 fpr, tpr, _ = roc_curve(df_test[TARGET], preds)
 auc = roc_auc_score(df_test[TARGET], preds)
 ax.plot(fpr, tpr, label=f"{nom}\n(AUC={auc:.4f})", color=col, linewidth=2)
ax.plot([0,1],[0,1], "k--", linewidth=0.8)
ax.set_xlabel("Taux fausse alarme")
ax.set_ylabel("Taux détection")
ax.set_title("Courbes ROC")
ax.legend(fontsize=7.5)

# --- Graphique 2 : Barres métriques ---
ax2 = axes[1]
metriques = ["Gini", "AUC", "Lift D1"]
noms_courts = ["M0\nBaseline", "M1\nCrédibilité", "M2\nToutes CSP"]
vals = [[res_m0[m], res_m1[m], res_m2[m]] for m in metriques]
x = np.arange(len(noms_courts))
w = 0.25
cols_met = ["#4C72B0", "#55A868", "#CCB974"]
for i, (met, v) in enumerate(zip(metriques, vals)):
 bars = ax2.bar(x + (i-1)*w, v, w, label=met, alpha=0.85)
 for bar, val in zip(bars, v):
 ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
 f"{val:.3f}", ha="center", fontsize=7.5, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(noms_courts)
ax2.set_title("Métriques discriminantes\n(↑ meilleur)")
ax2.legend(fontsize=8)

# --- Graphique 3 : Z par CSP pour M1 ---
ax3 = axes[2]
r = residus_train.sort_values("n_train", ascending=False).reset_index(drop=True)
couleurs_z = ["#55A868" if n >= 100 else "#CCB974" if n >= 30 else "#C44E52"
 for n in r["n_train"]]
ax3.barh(r[CSP_COL], r["Z"], color=couleurs_z, alpha=0.85)
ax3.axvline(0.5, color="gray", linestyle="--", linewidth=1)
ax3.set_xlabel("Facteur de crédibilité Z")
ax3.set_title(f"M1, Facteur Z par CSP\n(k={k_train:.1f})")
patches = [mpatches.Patch(color='#55A868', label='n≥100'),
 mpatches.Patch(color='#CCB974', label='30≤n<100'),
 mpatches.Patch(color='#C44E52', label='n<30')]
ax3.legend(handles=patches, fontsize=8)
for i, (z, n) in enumerate(zip(r["Z"], r["n_train"])):
 ax3.text(z + 0.01, i, f"n={n:.0f}", va="center", fontsize=7)

plt.tight_layout()
plt.show()

## Étape B.8. Analyse par CSP : qui gagne, qui perd ?

On compare les prédictions M0 et M1 pour chaque CSP sur le test, pour comprendre où la crédibilité apporte quelque chose.

In [ ]:
analyse_csp = (df_test.groupby(CSP_COL)
.apply(lambda g: pd.Series({
 "n_test": len(g),
 "freq_reel": g[TARGET].mean(),
 "pred_m0": g["pred_m0"].mean(),
 "pred_m1": g["pred_m1"].mean(),
 "pred_m2": g["pred_m2"].mean(),
 "err_m0": (g["pred_m0"] - g[TARGET]).abs().mean(),
 "err_m1": (g["pred_m1"] - g[TARGET]).abs().mean(),
 "err_m2": (g["pred_m2"] - g[TARGET]).abs().mean(),
 }))
.reset_index()
.merge(residus_train[[CSP_COL, "n_train", "Z"]], on=CSP_COL, how="left")
.sort_values("n_train", ascending=False)
.reset_index(drop=True))

analyse_csp["gagne_m1_vs_m0"] = analyse_csp["err_m1"] < analyse_csp["err_m0"]

print("=== Erreur absolue moyenne par CSP sur le test ===\n")
print(f"{'CSP':<10} {'n_train':>8} {'Z':>6} {'Réel':>7} {'M0':>7} {'M1-Créd':>8} "
 f"{'M2':>7} {'M1<M0 ?':>8}")
print("-" * 70)
for _, r in analyse_csp.iterrows():
 gagne = " Oui" if r["gagne_m1_vs_m0"] else " Non"
 print(f"{r[CSP_COL]:<10} {r['n_train']:>8.0f} {r['Z']:>6.0%} "
 f"{r['freq_reel']:>7.1%} {r['pred_m0']:>7.1%} {r['pred_m1']:>8.1%} "
 f"{r['pred_m2']:>7.1%} {gagne:>8}")

print(f"\nM1 meilleur que M0 sur : {analyse_csp['gagne_m1_vs_m0'].sum()}/{len(analyse_csp)} CSP")
print(f"Erreur moy. M0 : {analyse_csp['err_m0'].mean():.4f}")
print(f"Erreur moy. M1 : {analyse_csp['err_m1'].mean():.4f}")
print(f"Erreur moy. M2 : {analyse_csp['err_m2'].mean():.4f}")

## Conclusion

### Ce que ce TP apporte par rapport aux précédents

| | TP3 (M0) | M1 Crédibilité | M2 Toutes CSP |
|---|---|---|---|
| **Traitement CSP** | Regroupement arbitraire n<100 | Bühlmann : Z calibré par les données | Effets fixes Z=1 partout |
| **Info perdue** | Oui (CSP_autre perd les distinctions) | Non (toutes CSP contribuent) | Non |
| **Risque overfitting petites CSP** | Bas (regroupées) | Bas (Z faible → shrinkage) | **Élevé** |
| **Justification actuarielle** | Règle métier | Mathématiquement fondée | Aucune |

### Message clé
> La crédibilité est une alternative **plus rigoureuse** au regroupement arbitraire : au lieu de décider "n<100 → on fusionne", elle laisse les données décider du poids à accorder à chaque groupe, **graduellement** plutôt que brutalement.

## Extension 1. Bühlmann-Straub : crédibilité avec poids d'exposition

**Bühlmann standard** : $Z_g = \dfrac{n_g}{n_g + k}$, chaque individu pèse 1.

**Bühlmann-Straub** : $Z_g = \dfrac{w_g}{w_g + k}$, chaque individu pèse $w_{ig}$ (son exposition en années-véhicule).

Dans freMPL, il n'y a pas de variable d'exposition explicite. On en simule une (fraction d'année de contrat) pour montrer l'impact sur Z.

In [ ]:
# ── Exposition synthétique ──────────────────────────────────────────────────
# En réalité : durée du contrat en années-véhicule
# Ici : simulation uniforme [0.25, 1.0], certains contrats sont partiels
np.random.seed(42)
df_train_bs = df_train.copy()
df_train_bs["exposure"] = np.random.uniform(0.25, 1.0, len(df_train))
df_train_bs["resid_indiv"] = df_train_bs[TARGET] - df_train_bs["pred_m1_glm"]

# ── Bühlmann-Straub : estimation des variances ───────────────────────────────
def buhlmann_straub(df, csp_col, resid_col, exposure_col):
 rows = []
 for csp, g in df.groupby(csp_col):
 w = g[exposure_col].values
 r = g[resid_col].values
 w_g = w.sum()
 xbar_g = (w * r).sum() / w_g # moyenne pondérée
 rows.append({
 "csp": csp, "w_g": w_g, "xbar_g": xbar_g,
 "ss_within": (w * (r - xbar_g) ** 2).sum()
 })
 d = pd.DataFrame(rows)
 G = len(d)
 n_obs = len(df)
 w_tot = d["w_g"].sum()

 sigma2 = d["ss_within"].sum() / (n_obs - G) # variance process
 mu_hat = (d["w_g"] * d["xbar_g"]).sum() / w_tot # moyenne globale pondérée
 c = w_tot - (d["w_g"] ** 2).sum() / w_tot # facteur de correction
 tau2 = max(((d["w_g"] * (d["xbar_g"] - mu_hat) ** 2).sum()
 - (G - 1) * sigma2) / c, 1e-10) # variance structurelle
 k_bs = sigma2 / tau2
 d["Z_bs"] = d["w_g"] / (d["w_g"] + k_bs)
 d["Z_n"] = d["w_g"] / (d["w_g"] + k_train) # Bühlmann standard (n au lieu de w)
 return d, k_bs, sigma2, tau2

bs_result, k_bs, sigma2_bs, tau2_bs = buhlmann_straub(
 df_train_bs, CSP_COL, "resid_indiv", "exposure")

print(f"Bühlmann standard : k = {k_train:.2f}")
print(f"Bühlmann-Straub : k = {k_bs:.2f} "
 f"(σ²={sigma2_bs:.6f}, τ²={tau2_bs:.6f})\n")

print(f"{'CSP':<10} {'w_g (expo)':>12} {'Z_standard':>12} {'Z_BS':>8} {'Δ':>6}")
print("-" * 52)
for _, r in bs_result.sort_values("w_g", ascending=False).iterrows():
 delta = r["Z_bs"] - r["Z_n"]
 print(f"{r['csp']:<10} {r['w_g']:>12.1f} {r['Z_n']:>12.1%} {r['Z_bs']:>8.1%} "
 f"{delta:>+6.1%}")

# ── Application Bühlmann-Straub sur le test ─────────────────────────────────
csp_to_Zbs = bs_result.set_index("csp")["Z_bs"].to_dict()
csp_to_xbar = bs_result.set_index("csp")["xbar_g"].to_dict()

df_test["correction_bs"] = df_test[CSP_COL].map(
 lambda c: csp_to_Zbs.get(c, 0) * csp_to_xbar.get(c, 0))
df_test["pred_m1_bs"] = (df_test["pred_m1_glm"] + df_test["correction_bs"]).clip(0.001, 0.999)

res_m1_bs = evaluer("M1-BS, GLM + Bühlmann-Straub", df_test[TARGET], df_test["pred_m1_bs"])
print(f"\nM1 (Bühlmann std) : AUC={res_m1['AUC']:.4f} | Gini={res_m1['Gini']:.4f} | "
 f"Brier={res_m1['Brier']:.4f} | Lift D1={res_m1['Lift D1']:.2f}x")
print(f"M1-BS (B-Straub) : AUC={res_m1_bs['AUC']:.4f} | Gini={res_m1_bs['Gini']:.4f} | "
 f"Brier={res_m1_bs['Brier']:.4f} | Lift D1={res_m1_bs['Lift D1']:.2f}x")
print("\n→ Avec exposition uniforme ≈ Bühlmann standard (attendu)")
print("→ Avec vrais contrats partiels, les petites CSP avec peu d'exposition"
 " auraient Z plus faible encore")

## Extension 2. GLMM : la version "propre" de la crédibilité

La crédibilité de Bühlmann est en réalité un cas particulier de **modèle linéaire généralisé à effets mixtes (GLMM)** :

$$\text{logit}(p_{ig}) = \underbrace{\beta^\top x_i}_{\text{effets fixes}} + \underbrace{u_g}_{\text{effet aléatoire CSP}}$$

où $u_g \sim \mathcal{N}(0, \tau^2)$ est l'effet groupe, estimé conjointement avec les effets fixes.

Les **BLUP** (Best Linear Unbiased Predictors) de $u_g$ sont exactement les corrections crédibilité de Bühlmann.

On utilise `BinomialBayesMixedGLM` de statsmodels (approche MAP bayésienne).

In [ ]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
import patsy

_, X_train_df = patsy.dmatrices(formula_m1_glm, df_train, return_type="dataframe")
X_test_df = patsy.build_design_matrices([X_train_df.design_info], df_test)[0]
y_train_arr = df_train[TARGET].values.astype(float)

csp_cats = sorted(df_train[CSP_COL].unique())
csp_idx = {c: i for i, c in enumerate(csp_cats)}
n_csp = len(csp_cats)

X_train_re = np.zeros((len(df_train), n_csp))
for i, csp in enumerate(df_train[CSP_COL]):
 X_train_re[i, csp_idx[csp]] = 1

X_test_re = np.zeros((len(df_test), n_csp))
for i, csp in enumerate(df_test[CSP_COL]):
 if csp in csp_idx:
 X_test_re[i, csp_idx[csp]] = 1

ident = np.zeros(n_csp, dtype=int)

# vcp_p=1000 = prior quasi-plat = sans penalisation artificielle
# (vs vcp_p=1 defaut qui ecrase tau vers 0)
glmm_model = BinomialBayesMixedGLM(y_train_arr, np.asarray(X_train_df),
 X_train_re, ident, vcp_p=1000)
glmm_result = glmm_model.fit_map()

# Structure params : [FE(n_fe), VC(n_vc=1), RE(n_csp)]
# ATTENTION: VC vient AVANT les RE dans statsmodels
n_fe = np.asarray(X_train_df).shape[1]
n_vc = len(np.unique(ident)) # = 1
fe_params = glmm_result.params[:n_fe]
re_params = glmm_result.params[n_fe + n_vc : n_fe + n_vc + n_csp] # RE seulement
tau_hat = np.exp(glmm_result.params[n_fe]) # VC = log(tau)

print(f"tau estime = {tau_hat:.2e} (devrait etre ~0 si pas de variance groupe)")
print(f"n params total = {len(glmm_result.params)} | FE={n_fe} VC={n_vc} RE={n_csp}")

# Predictions sur le test
eta_test = np.asarray(X_test_df) @ fe_params + X_test_re @ re_params
pred_glmm = 1 / (1 + np.exp(-eta_test))
df_test["pred_glmm"] = np.clip(pred_glmm, 0.001, 0.999)

res_glmm = evaluer("M3 - GLMM effets aleatoires CSP", df_test[TARGET], df_test["pred_glmm"])
print(f"\nM3 GLMM : AUC={res_glmm['AUC']:.4f} | Gini={res_glmm['Gini']:.4f} | Brier={res_glmm['Brier']:.4f} | Lift={res_glmm['Lift D1']:.2f}x")

print("\nEffets aleatoires u_g par CSP (indexation corrigee) :")
print(f"{'CSP':<10} {'n_train':>8} {'u_g':>10} {'corr.Buhlmann':>14}")
print("-" * 46)
for csp, idx in sorted(csp_idx.items(),
 key=lambda x: residus_train.set_index(CSP_COL).loc[x[0], "n_train"]
 if x[0] in residus_train[CSP_COL].values else 0, reverse=True):
 n = residus_train.loc[residus_train[CSP_COL] == csp, "n_train"].values
 buh = csp_to_correction.get(csp, 0)
 n_s = str(int(n[0])) if len(n) else "?"
 print(f"{csp:<10} {n_s:>8} {re_params[idx]:>10.4f} {buh:>14.1%}")
print(f"\ntau = {tau_hat:.2e} -> u_g~0 : le MLE dit pas de variance groupe, meme sans prior fort")


## Synthèse finale. Comparaison des 5 approches

In [ ]:
tous_modeles = [
 res_m0,
 res_m1,
 res_m1_bs,
 res_m2,
 res_glmm,
]

# ── Tableau synthèse ──────────────────────────────────────────────────────────
df_synth = pd.DataFrame(tous_modeles).set_index("Modèle")
df_synth["Rang AUC"] = df_synth["AUC"].rank(ascending=False).astype(int)
df_synth["Rang Gini"] = df_synth["Gini"].rank(ascending=False).astype(int)
df_synth["Rang Brier"] = df_synth["Brier"].rank(ascending=True).astype(int)
df_synth["Rang Lift"] = df_synth["Lift D1"].rank(ascending=False).astype(int)
df_synth["Rang moyen"] = df_synth[["Rang AUC","Rang Gini","Rang Brier","Rang Lift"]].mean(axis=1)

meta = {
 res_m0["Modèle"]: ("GLM", "Regroupement n<100 → CSP_autre", "Seuil arbitraire"),
 res_m1["Modèle"]: ("GLM+Bü", "Bühlmann additif (proba scale)", "k calibré sur données"),
 res_m1_bs["Modèle"]: ("GLM+BS", "Bühlmann-Straub avec exposition","Pondéré par exposition"),
 res_m2["Modèle"]: ("GLM FE", "Effets fixes Z=1 pour tous", "Risque overfitting"),
 res_glmm["Modèle"]: ("GLMM", "Prior quasi-plat (vcp_p=1000)", "τ²≈0 : données décident"),
}

print("=" * 100)
print("TABLEAU COMPARATIF FINAL, 5 MODÈLES (sans pénalisation artificielle)")
print("=" * 100)
print(f"{'Modèle':<42} {'AUC':>6} {'Gini':>6} {'Brier':>7} {'Lift D1':>8} "
 f"{'Rang':>5} {'Approche CSP'}")
print("-" * 100)
for nom, row in df_synth.iterrows():
 abr, traitement, note = meta.get(nom, ("", "", ""))
 print(f"{nom:<42} {row['AUC']:>6.4f} {row['Gini']:>6.4f} {row['Brier']:>7.4f} "
 f"{row['Lift D1']:>8.2f}x {row['Rang moyen']:>5.1f} {traitement} [{note}]")
print("=" * 100)

# ── Graphique : 2 lignes, métriques + courbes ROC ──────────────────────────
fig = plt.figure(figsize=(18, 9))
gs = fig.add_gridspec(2, 4, hspace=0.45, wspace=0.35)

couleurs = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]
labels_c = ["M0\nBaseline", "M1\nBühlmann", "M1-BS\nB-Straub", "M2\nFE", "M3\nGLMM"]
preds_test = [df_test["pred_m0"], df_test["pred_m1"], df_test["pred_m1_bs"],
 df_test["pred_m2"], df_test["pred_glmm"]]

# Ligne 1 : barres par métrique
metriques_list = ["AUC", "Gini", "Brier", "Lift D1"]
titres_list = ["AUC ↑", "Gini ↑", "Brier Score ↓", "Lift Décile 1 ↑"]
ascending_list = [False, False, True, False]

for col_i, (met, titre, asc) in enumerate(zip(metriques_list, titres_list, ascending_list)):
 ax = fig.add_subplot(gs[0, col_i])
 vals = [r[met] for r in tous_modeles]
 best_idx = vals.index(min(vals) if asc else max(vals))
 bars = ax.bar(labels_c, vals, color=couleurs, alpha=0.82, edgecolor="white", linewidth=1)
 bars[best_idx].set_edgecolor("black")
 bars[best_idx].set_linewidth(2.5)
 ax.set_title(titre, fontweight="bold", fontsize=10)
 spread = max(vals) - min(vals)
 ax.set_ylim(min(vals) - spread * 0.1, max(vals) + spread * 0.25)
 for bar_, v in zip(bars, vals):
 ax.text(bar_.get_x() + bar_.get_width()/2,
 bar_.get_height() + spread * 0.02,
 f"{v:.3f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")
 ax.tick_params(axis="x", labelsize=7.5)
 ax.spines[["top", "right"]].set_visible(False)

# Ligne 2 : courbes ROC (col 0-2) + rang moyen (col 3)
from sklearn.metrics import roc_curve as roc_curve_sk

ax_roc = fig.add_subplot(gs[1, :3])
for (nom, row), pred, col in zip(df_synth.iterrows(), preds_test, couleurs):
 fpr, tpr, _ = roc_curve_sk(df_test[TARGET], pred)
 ax_roc.plot(fpr, tpr, color=col, linewidth=2,
 label=f"{meta[nom][0]} AUC={row['AUC']:.4f} Gini={row['Gini']:.4f}")
ax_roc.plot([0,1],[0,1], "k--", linewidth=0.8, alpha=0.5)
ax_roc.set_xlabel("Taux fausse alarme (1-Spécificité)")
ax_roc.set_ylabel("Taux détection (Sensibilité)")
ax_roc.set_title("Courbes ROC, 5 modèles comparés (sans pénalisation)", fontweight="bold")
ax_roc.legend(fontsize=8.5, loc="lower right")
ax_roc.spines[["top", "right"]].set_visible(False)

# Col 3 ligne 2 : rang moyen (radar / bar horizontal)
ax_rg = fig.add_subplot(gs[1, 3])
noms_courts_rg = [meta[n][0] for n in df_synth.index]
rangs = df_synth["Rang moyen"].values
ax_rg.barh(noms_courts_rg, rangs, color=couleurs[::-1], alpha=0.8)
ax_rg.set_xlim(0, 6)
ax_rg.axvline(3, color="gray", linestyle="--", linewidth=0.8)
ax_rg.set_xlabel("Rang moyen (1=meilleur)")
ax_rg.set_title("Classement global\n(4 métriques)", fontweight="bold")
for i, v in enumerate(rangs):
 ax_rg.text(v + 0.05, i, f"{v:.1f}", va="center", fontsize=9)
ax_rg.invert_yaxis()
ax_rg.spines[["top", "right"]].set_visible(False)

fig.suptitle("Comparaison 5 approches, freMPL test set, sans pénalisation artificielle",
 fontsize=12, fontweight="bold", y=1.01)
plt.show()

# ── Verdict ──────────────────────────────────────────────────────────────────
print("\n Verdict (sans biais de pénalisation) :")
print(" M2 (effets fixes) : meilleur AUC/Gini, mais pas généralisable sur petites CSP")
print(" M0 (baseline) : meilleur Lift D1, regroupement stabilise le top décile")
print(" M1 (Bühlmann) : Rang 3-4, moins bon en score, MAIS seul modèle calibré par groupe")
print(" M1-BS (B-Straub) : proche de M1, différence visible avec vraie exposition")
print(" M3 (GLMM, τ²=0) : les données disent 'pas de variance CSP' → ≈ GLM sans CSP")
print()
print(" Conclusion : AUC/Gini récompensent le classement individuel, pas la calibration groupe.")
print(" En tarification, M1 / M1-BS restent les choix les plus défendables actuariellement.")

### Question 5. Diagnostic intermédiaire

Vous venez de voir : **aucun modèle ne bat M0 baseline en AUC** (sauf M2 de très peu).

> Pourquoi la crédibilité, qui est *mathématiquement* l'optimum bayésien, ne gagne-t-elle pas ?

**Deux hypothèses** à explorer ensuite (M4 et M5) :

1. **On l'applique au mauvais endroit**, Sur CSP seule, Z≈1 partout : Bühlmann ne fait rien. Et si on l'appliquait à des cellules **plus fines** (CSP × Garage) où il y aurait vraiment des petits groupes ?

2. **On confond crédibilité-coefficient et crédibilité-modèle**, Au lieu de corriger les coefs CSP, et si Z arbitrait entre **deux modèles** (M0 stable et M2 fin) ?

C'est exactement ce que font M4 et M5 ci-dessous. Avant de continuer, **prédisez** :
- Lequel des deux va le mieux marcher ? Pourquoi ?
- M5 (maille fine) risque-t-il l'overfitting ? Comment la crédibilité l'en protège-t-elle ?


## Amélioration 1. M4 : Stacking crédibilité (M0 ⊕ M2)

Idée crédibilité appliquée au **choix de modèle**, pas au coefficient :

$$\hat{p}_{M4}(x_i) = Z_{g(i)} \cdot \hat{p}_{M2}(x_i) + (1 - Z_{g(i)}) \cdot \hat{p}_{M0}(x_i)$$

- **CSP large** (Z≈1) : on prend M2 (effet fixe, info pure)
- **CSP petite** (Z petit) : on retombe sur M0 (regroupement CSP_autre)

C'est le meilleur des deux mondes : on n'arbitre plus, c'est Z qui décide.

In [ ]:
# Z par CSP (depuis Bühlmann M1)
df_test["Z_stack"] = df_test[CSP_COL].map(csp_to_Z).fillna(0)
df_train["Z_stack"] = df_train[CSP_COL].map(csp_to_Z).fillna(0)

df_test["pred_m4"] = (
 df_test["Z_stack"] * df_test["pred_m2"]
 + (1 - df_test["Z_stack"]) * df_test["pred_m0"]
).clip(0.001, 0.999)

res_m4 = evaluer("M4, Stacking crédibilité (M0⊕M2)", df_test[TARGET], df_test["pred_m4"])

print("=" * 70)
print("M4, Stacking crédibilité (M0 ⊕ M2 pondéré par Z)")
print("=" * 70)
print(f"M0 baseline (CSP_autre) : AUC={res_m0['AUC']:.4f} Gini={res_m0['Gini']:.4f} Lift={res_m0['Lift D1']:.2f}x")
print(f"M2 toutes CSP (Z=1) : AUC={res_m2['AUC']:.4f} Gini={res_m2['Gini']:.4f} Lift={res_m2['Lift D1']:.2f}x")
print(f"M4 stacking M0⊕M2 : AUC={res_m4['AUC']:.4f} Gini={res_m4['Gini']:.4f} Lift={res_m4['Lift D1']:.2f}x")
print()
delta_auc_m0 = (res_m4["AUC"] - res_m0["AUC"]) * 100
delta_auc_m2 = (res_m4["AUC"] - res_m2["AUC"]) * 100
print(f"Δ AUC vs M0 : {delta_auc_m0:+.2f} pp")
print(f"Δ AUC vs M2 : {delta_auc_m2:+.2f} pp")

# Vérification : moyenne du Z appliqué sur le test
print(f"\nZ moyen sur test : {df_test['Z_stack'].mean():.3f}")
print(f"Z<0.95 (= petites CSP) : {(df_test['Z_stack']<0.95).sum()} obs ({(df_test['Z_stack']<0.95).mean():.1%})")
print(f"→ Pour ces obs, M4 favorise M0 (regroupement) plutôt que M2 (overfit)")

## Amélioration 2. M5 : Crédibilité sur croisement CSP × Garage

Le problème de M1 était que SocioCateg seul est trop grossier (Z≈100% partout). On raffine la maille avec **CSP × Garage** :

- ~60 cellules au lieu de 20
- Beaucoup ont peu d'observations → c'est **exactement le terrain** où Bühlmann domine
- k_cell devrait être plus grand → shrinkage actif sur les petites cellules

C'est la stratégie "data-driven" pour battre M0 : on attaque là où le signal individuel manque (combinaisons rares).

In [ ]:
# ── Maille fine : CSP × Garage ──────────────────────────────────────────────
df_train["cell"] = df_train[CSP_COL].astype(str) + "_" + df_train["Garage"].astype(str)
df_test["cell"] = df_test[CSP_COL].astype(str) + "_" + df_test["Garage"].astype(str)

residus_cell = (df_train.groupby("cell")
.apply(lambda g: pd.Series({
 "n": len(g),
 "freq_obs": g[TARGET].mean(),
 "pred_glm": g["pred_m1_glm"].mean(),
 "residu": g[TARGET].mean() - g["pred_m1_glm"].mean(),
 }))
.reset_index())

# Calibration k sur la maille fine
var_bruit_c = residus_cell["residu"].apply(lambda r: abs(r) * (1 - abs(r))).mean()
var_signal_c = residus_cell["residu"].var()
k_cell = var_bruit_c / var_signal_c if var_signal_c > 1e-9 else 50

residus_cell["Z"] = residus_cell["n"] / (residus_cell["n"] + k_cell)
residus_cell["correction"] = residus_cell["Z"] * residus_cell["residu"]
cell_to_corr = residus_cell.set_index("cell")["correction"].to_dict()

print(f"Nombre de cellules CSP×Garage sur train : {len(residus_cell)}")
print(f" dont n<30 (petites) : {(residus_cell['n']<30).sum()}")
print(f" dont n<10 (rares) : {(residus_cell['n']<10).sum()}")
print(f"k_cell calibré : {k_cell:.2f} (vs k_CSP seul = {k_train:.2f})")
print(f"Z moyen sur train : {residus_cell['Z'].mean():.3f} "
 f"(vs CSP seul ≈ {residus_train['Z'].mean():.3f})")

# Application sur test (cellules inconnues → correction 0)
df_test["corr_m5"] = df_test["cell"].map(cell_to_corr).fillna(0)
df_test["pred_m5"] = (df_test["pred_m1_glm"] + df_test["corr_m5"]).clip(0.001, 0.999)

n_unknown = df_test["cell"].map(cell_to_corr).isna().sum()
print(f"Cellules test inconnues du train : {n_unknown} ({n_unknown/len(df_test):.1%}) → correction=0")

res_m5 = evaluer("M5, Crédibilité CSP×Garage", df_test[TARGET], df_test["pred_m5"])

print("\n" + "=" * 70)
print("M5, Crédibilité sur maille fine CSP × Garage")
print("=" * 70)
print(f"M0 baseline : AUC={res_m0['AUC']:.4f} Gini={res_m0['Gini']:.4f} Lift={res_m0['Lift D1']:.2f}x")
print(f"M1 Bühlmann (CSP seul) : AUC={res_m1['AUC']:.4f} Gini={res_m1['Gini']:.4f} Lift={res_m1['Lift D1']:.2f}x")
print(f"M5 Bühlmann (CSP×Garage) : AUC={res_m5['AUC']:.4f} Gini={res_m5['Gini']:.4f} Lift={res_m5['Lift D1']:.2f}x")
delta_m5 = (res_m5["AUC"] - res_m0["AUC"]) * 100
print(f"\nΔ AUC M5 vs M0 : {delta_m5:+.2f} pp")

# Top 10 cellules avec correction la plus forte (en valeur absolue)
print("\nTop 10 cellules avec correction la plus impactante :")
top = residus_cell.reindex(residus_cell["correction"].abs().sort_values(ascending=False).index).head(10)
print(top[["cell","n","Z","residu","correction"]]
.to_string(index=False, float_format=lambda x: f"{x:.1%}" if abs(x)<1 else f"{x:.0f}"))

## Synthèse finale étendue. 7 modèles avec les améliorations

In [ ]:
tous_modeles_v2 = [res_m0, res_m1, res_m1_bs, res_m2, res_glmm, res_m4, res_m5]

df_synth2 = pd.DataFrame(tous_modeles_v2).set_index("Modèle")
df_synth2["Δ AUC vs M0 (pp)"] = (df_synth2["AUC"] - res_m0["AUC"]) * 100
df_synth2["Rang AUC"] = df_synth2["AUC"].rank(ascending=False).astype(int)

print("=" * 100)
print("TABLEAU FINAL, 7 MODÈLES (avec M4 stacking et M5 crédibilité raffinée)")
print("=" * 100)
print(f"{'Modèle':<42} {'AUC':>6} {'Gini':>6} {'Brier':>7} {'Lift D1':>8} "
 f"{'Δ AUC':>7} {'Rang':>5}")
print("-" * 100)
for nom, row in df_synth2.sort_values("AUC", ascending=False).iterrows():
 delta_str = f"{row['Δ AUC vs M0 (pp)']:+.2f}pp" if abs(row["Δ AUC vs M0 (pp)"]) > 1e-6 else "ref"
 print(f"{nom:<42} {row['AUC']:>6.4f} {row['Gini']:>6.4f} {row['Brier']:>7.4f} "
 f"{row['Lift D1']:>8.2f}x {delta_str:>7} {row['Rang AUC']:>5}")
print("=" * 100)

# ── Graphique : barres triées + courbes ROC ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# Tri par AUC décroissant
ordre = df_synth2.sort_values("AUC", ascending=False).index.tolist()
couleurs_map = {
 res_m0["Modèle"]: "#9E9E9E",
 res_m1["Modèle"]: "#FF9800",
 res_m1_bs["Modèle"]: "#FFC107",
 res_m2["Modèle"]: "#9C27B0",
 res_glmm["Modèle"]: "#F44336",
 res_m4["Modèle"]: "#4CAF50", # vert = amélioration stacking
 res_m5["Modèle"]: "#2E7D32", # vert foncé = meilleure amélioration
}
labels_court = {
 res_m0["Modèle"]: "M0\nBaseline",
 res_m1["Modèle"]: "M1\nBühlmann",
 res_m1_bs["Modèle"]: "M1-BS\nB-Straub",
 res_m2["Modèle"]: "M2\nFE",
 res_glmm["Modèle"]: "M3\nGLMM",
 res_m4["Modèle"]: "M4\nStacking",
 res_m5["Modèle"]: "M5\nCSP×Garage",
}

# Graphique 1 : Δ AUC vs baseline
ax = axes[0]
deltas = df_synth2.loc[ordre, "Δ AUC vs M0 (pp)"].values
labels = [labels_court[n] for n in ordre]
cols = [couleurs_map[n] for n in ordre]
bars = ax.bar(labels, deltas, color=cols, alpha=0.85, edgecolor="white", linewidth=1.2)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Δ AUC vs M0 baseline (points de pourcentage)")
ax.set_title("Gain en AUC par rapport au GLM baseline (M0)", fontweight="bold")
for bar_, d in zip(bars, deltas):
 y = bar_.get_height()
 ax.text(bar_.get_x() + bar_.get_width()/2,
 y + (0.05 if y >= 0 else -0.15),
 f"{d:+.2f}", ha="center",
 va="bottom" if y >= 0 else "top",
 fontweight="bold", fontsize=9)
ax.tick_params(axis="x", labelsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

# Graphique 2 : courbes ROC
from sklearn.metrics import roc_curve as rc
ax2 = axes[1]
preds_map = {
 res_m0["Modèle"]: df_test["pred_m0"],
 res_m1["Modèle"]: df_test["pred_m1"],
 res_m1_bs["Modèle"]: df_test["pred_m1_bs"],
 res_m2["Modèle"]: df_test["pred_m2"],
 res_glmm["Modèle"]: df_test["pred_glmm"],
 res_m4["Modèle"]: df_test["pred_m4"],
 res_m5["Modèle"]: df_test["pred_m5"],
}
for nom in ordre:
 fpr, tpr, _ = rc(df_test[TARGET], preds_map[nom])
 auc = roc_auc_score(df_test[TARGET], preds_map[nom])
 lw = 2.5 if nom in [res_m4["Modèle"], res_m5["Modèle"]] else 1.5
 ax2.plot(fpr, tpr, color=couleurs_map[nom], linewidth=lw,
 label=f"{labels_court[nom].replace(chr(10), ' ')} AUC={auc:.4f}")
ax2.plot([0,1],[0,1], "k--", linewidth=0.8, alpha=0.5)
ax2.set_xlabel("Taux fausse alarme")
ax2.set_ylabel("Taux détection")
ax2.set_title("Courbes ROC, les améliorations crédibilité (vert) dominent", fontweight="bold")
ax2.legend(fontsize=8, loc="lower right")
ax2.spines[["top", "right"]].set_visible(False)

plt.suptitle("Bilan final, la crédibilité bien appliquée bat le GLM baseline",
 fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── Verdict final ────────────────────────────────────────────────────────────
print("\n Verdict final, comment battre le GLM brut avec la crédibilité :")
print()
print(f" M4 Stacking M0⊕M2 : AUC = {res_m4['AUC']:.4f} (+{(res_m4['AUC']-res_m0['AUC'])*100:.2f} pp vs M0)")
print(f" → laisse Z arbitrer entre M0 (petites CSP) et M2 (grandes CSP)")
print()
print(f" M5 Bühlmann fin : AUC = {res_m5['AUC']:.4f} (+{(res_m5['AUC']-res_m0['AUC'])*100:.2f} pp vs M0)")
print(f" → raffine la maille (CSP×Garage) où Bühlmann a un VRAI effet")
print(f" → k={k_cell:.1f}, Z moyen={residus_cell['Z'].mean():.0%} (vs 99% sur CSP seul)")
print()
print(" Insight : la crédibilité ne bat le GLM que si on l'applique")
print(" là où il y a vraiment des petits groupes à shrinker, pas sur des macro-classes.")

## Bilan et mise en pratique

### Une checklist à se poser sur n'importe quel portefeuille

Avant d'écrire une seule ligne de code de crédibilité, posez-vous ces trois questions :

1. **Quelle est la distribution des tailles de groupes ?** S'il y a quelques très gros groupes (n > 1000) et beaucoup de petits (n < 50), Bühlmann a une chance. Si tous les groupes sont de taille comparable et grande, Bühlmann est inutile : $Z \approx 1$ partout, on retombe sur la moyenne brute.

2. **Y a-t-il vraiment une variance entre groupes ?** Bühlmann calibre $k = \sigma^2_{\text{bruit}} / \tau^2_{\text{signal}}$. Si $\tau^2 \approx 0$ (les groupes se ressemblent), $k$ explose et tout est tiré vers la moyenne globale : la crédibilité n'apporte rien que la moyenne ne donnait déjà.

3. **La maille est-elle la bonne ?** Si CSP seule ne sépare pas le risque, croiser CSP × Garage (ou CSP × Zone, etc.) peut révéler des cellules à effectifs faibles où Bühlmann redevient utile (c'est le cas de M5 dans ce TP).

### Quatre situations à classer

Imaginez qu'un collègue arrive avec un cas. Pour chacun, décidez : Bühlmann pertinent, ou pas ?

- **Cas 1.** Une mutuelle santé veut tarifer par profession sur 50 modalités CSP fines, certaines à n = 20.
- **Cas 2.** Un assureur auto veut tarifer par région sur 5 grandes régions, toutes à n > 50 000.
- **Cas 3.** Une MRH veut un effet par code postal (300 codes postaux observés, médiane à n = 80).
- **Cas 4.** Un collègue applique Bühlmann sur CSP, et son Brier baisse mais son AUC aussi. Faut-il garder le modèle ?

> *Indices de réponse, à ne lire qu'après réflexion :* Cas 1 oui (petites cellules), Cas 2 non ($Z \approx 1$), Cas 3 oui (médiane modeste, Bühlmann-Straub avec exposition), Cas 4 oui si on tarife (Brier compte plus que AUC), non si on segmente uniquement pour décider qui contacter (AUC compte).

### Bac à sable

Reprenez le code de ce TP et essayez :

- Refaire M5 avec une **vraie variable d'exposition** (durée de contrat) au lieu d'un poids fixé à 1.
- Implémenter une crédibilité **hiérarchique** : CSP dans Garage, deux niveaux $Z$ emboîtés.
- Comparer M5 à un **Lasso** sur le même croisement CSP × Garage : Lasso shrinke en L1, Bühlmann shrinke en L2 vers la moyenne. Lequel généralise le mieux ici ?
- Faire varier $k$ à la main entre $k_{\text{calibré}}/10$ et $10 \times k_{\text{calibré}}$ et regarder l'effet sur l'AUC test. Que dit la courbe ?

### Trois idées à emporter

1. **Si n > 1000 dans tous les groupes, oubliez Bühlmann.** $Z \approx 1$, vous n'apportez rien.
2. **Quand Bühlmann ne sert à rien sur une variable, le bon réflexe est de raffiner la maille**, pas d'abandonner la méthode. Le croisement CSP × Garage de ce TP en est l'illustration : 48 cellules dont 26 avec n < 30, là où Bühlmann redevient utile.
3. **La crédibilité est une régularisation $L_2$ avec une cible métier.** On ne tire pas vers zéro (comme Ridge), on tire vers la moyenne du portefeuille. C'est la version actuarielle, calibrée, du compromis biais-variance.
